# Taller 1. Formulación y resolución de un modelo en red

Investigación de Operaciones
Doctorado en Ingeniería, consorcio UV–UTA
Jueves 10 de septiembre de 2026, Bloque B

---

**Integrante:** Paul Eduardo Villalón Meriches
**Instancia asignada:** Equipo D — Industria 4.0: Asignación de órdenes a celdas robotizadas
**Fecha:** Septiembre de 2026

---

Este cuaderno resuelve integralmente la Instancia D (Industria 4.0) siguiendo la plantilla oficial y las exigencias comunes de la rúbrica (formulación matemática explícita, lectura dinámica de CSV sin datos incrustados, verificación de conservación de flujo, verificación independiente por segunda vía, demostración de total unimodularidad, análisis dual con unidades y rangos de validez, extensión con restricción compartida LP vs. MILP, análisis de sensibilidad y limitaciones del modelo).

> **Regla dura respetada:** ningún dato numérico del problema está escrito dentro del modelo; todos los parámetros se leen dinámicamente desde `tiempos.csv` y `restriccion_adicional.csv`.

## Paso 0. Verificación del entorno

In [4]:
import sys
print("Python:", sys.version.split()[0])

import pyomo.environ as pyo
print("Pyomo:", pyo.__version__ if hasattr(pyo, "__version__") else "instalado")

solver = pyo.SolverFactory("appsi_highs")
print("HiGHS (appsi_highs) disponible:", solver.available())

import pandas as pd
import numpy as np
import scipy
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| scipy:", scipy.__version__)

Python: 3.13.18
Pyomo: instalado
HiGHS (appsi_highs) disponible: True
pandas: 2.2.3 | numpy: 2.4.6 | scipy: 1.16.3


## Paso 1. Lectura de los datos

Los archivos de entrada de la Instancia D (`tiempos.csv` y `restriccion_adicional.csv`) utilizan punto y coma (`;`) como separador. El código busca primero en la carpeta `datos/` (estructura estándar del repositorio) y, si no existe, en el directorio actual.

In [5]:
from pathlib import Path
import pandas as pd

# Busca en datos/ según la estructura del repositorio; si no existe, usa el directorio actual
CARPETA = Path("datos") if (Path("datos") / "tiempos.csv").exists() else Path(".")
print("Leyendo datos desde:", CARPETA.resolve())

df_tiempos = pd.read_csv(CARPETA / "tiempos.csv", sep=";")
df_restriccion = pd.read_csv(CARPETA / "restriccion_adicional.csv", sep=";")

print("\n--- tiempos.csv (minutos por asignación orden -> celda) ---")
display(df_tiempos)


Leyendo datos desde: /content/datos

--- tiempos.csv (minutos por asignación orden -> celda) ---


,orden,C1,C2,C3,C4,C5,C6
0,O1,42,55,61,38,70,49
1,O2,58,40,44,63,52,47
2,O3,36,62,39,51,58,66
3,O4,49,43,57,45,41,53
4,O5,64,51,46,59,37,42
5,O6,45,60,50,44,55,38


## Paso 2. Estructura de la red y formulación matemática explícita (Pregunta 1)

### 2.1 Red subyacente, nodos, arcos y unidades

El problema de asignación de órdenes a celdas robotizadas es un caso particular del problema de flujo a costo mínimo sobre un grafo dirigido bipartito completo $G = (N, A)$:

* Conjuntos:
  * Conjunto de nodos origen (órdenes de producción): $O = \{O_1, O_2, O_3, O_4, O_5, O_6\}$, con $|O| = 6$.
  * Conjunto de nodos destino (celdas robotizadas): $C = \{C_1, C_2, C_3, C_4, C_5, C_6\}$, con $|C| = 6$.
  * Conjunto total de nodos: $N = O \cup C$ ($|N| = 12$).
  * Conjunto de arcos dirigidos: $A = O \times C = \{(i, j) : i \in O,\; j \in C\}$ ($|A| = 36$), donde el arco $(i, j)$ representa asignar la orden $i$ a la celda $j$.

* Parámetros (leídos desde los CSV):
  * $c_{ij}$: tiempo total de preparación y proceso al asignar la orden $i \in O$ a la celda $j \in C$, medido en minutos por asignación (`tiempos.csv`).
  * $q_n$: flujo exógeno del nodo $n \in N$, medido en órdenes:
    * $q_i = +1$ orden para todo nodo origen $i \in O$ (cada orden debe salir exactamente una vez hacia alguna celda).
    * $q_j = -1$ orden para todo nodo destino $j \in C$ (cada celda debe recibir exactamente una orden).
  * Cotas de arco: $l_{ij} = 0$ y $u_{ij} = 1$ (en unidades de fracción de asignación, adimensional / órdenes) para todo $(i, j) \in A$.
  * Para la Parte 2 (`restriccion_adicional.csv`): subconjunto de celdas con brazo compartido $C_{\text{brazo}} = \{C_1, C_2\}$ y tope conjunto $T = 77$ minutos.

* Variables de decisión:
  * $x_{ij} \in [0, 1]$: fracción de la orden $i \in O$ asignada a la celda $j \in C$ (unidades: órdenes; cuando resulta entera, $x_{ij} = 1$ indica que la orden $i$ se ejecuta en la celda $j$ y $x_{ij} = 0$ en caso contrario).

### 2.2 Formulación del modelo base (Flujo a costo mínimo)

$$\min \; z = \sum_{i \in O} \sum_{j \in C} c_{ij}\,x_{ij} \quad [\text{minutos}]$$

sujeto a:

1. Conservación de flujo en nodos origen (cada orden se asigna a exactamente una celda):
   $$\sum_{j \in C} x_{ij} = +1 \qquad \forall i \in O$$
2. Conservación de flujo en nodos destino (cada celda recibe exactamente una orden; en convención $\text{sale} - \text{entra} = q_j$):
   $$-\sum_{i \in O} x_{ij} = -1 \iff \sum_{i \in O} x_{ij} = 1 \qquad \forall j \in C$$
3. Cotas de capacidad (relajación lineal continua):
   $$0 \le x_{ij} \le 1 \qquad \forall (i, j) \in A$$

### 2.3 Justificación de igualdades frente a desigualdades

Todas las restricciones nodales se formulan con igualdad ($=$) porque la oferta total de órdenes ($\sum_{i \in O} q_i = +6$) coincide exactamente con la demanda total de celdas ($\sum_{j \in C} |q_j| = 6$), verificando la condición necesaria de factibilidad $\sum_{n \in N} q_n = 0$. Además, el enunciado exige que se procesen las 6 órdenes asignando exactamente una orden por celda; no existe oferta excedente ni celdas ociosas que requieran desigualdades o nodos ficticios.

In [6]:
# Construcción dinámica de la red bipartita desde tiempos.csv
ordenes = list(df_tiempos["orden"])
celdas = [col for col in df_tiempos.columns if col != "orden"]

N = ordenes + celdas

# Flujo exógeno q_i: +1 para cada orden (oferta), -1 para cada celda (demanda)
q = {o: 1.0 for o in ordenes}
q.update({c: -1.0 for c in celdas})

# Arcos A: (orden, celda) -> (costo_minutos, cota_inferior, cota_superior)
A = {
    (row["orden"], c): (float(row[c]), 0.0, 1.0)
    for _, row in df_tiempos.iterrows()
    for c in celdas
}

# Parámetros de la restricción adicional leídos desde restriccion_adicional.csv
celdas_brazo = [c.strip() for c in str(df_restriccion.loc[0, "celdas"]).split("+")]
tope_brazo = float(df_restriccion.loc[0, "tope_minutos"])
display (N)
display (q)
display (A)
print(f"Nodos ({len(N)}): {N}")
print(f"Arcos ({len(A)}): red bipartita completa {len(ordenes)} x {len(celdas)}")
print("Suma de los flujos exógenos (debe ser 0):", sum(q.values()))
assert abs(sum(q.values())) < 1e-9, "La suma de flujos exógenos debe ser cero."

['O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6']

{'O1': 1.0,
 'O2': 1.0,
 'O3': 1.0,
 'O4': 1.0,
 'O5': 1.0,
 'O6': 1.0,
 'C1': -1.0,
 'C2': -1.0,
 'C3': -1.0,
 'C4': -1.0,
 'C5': -1.0,
 'C6': -1.0}

{('O1', 'C1'): (42.0, 0.0, 1.0),
 ('O1', 'C2'): (55.0, 0.0, 1.0),
 ('O1', 'C3'): (61.0, 0.0, 1.0),
 ('O1', 'C4'): (38.0, 0.0, 1.0),
 ('O1', 'C5'): (70.0, 0.0, 1.0),
 ('O1', 'C6'): (49.0, 0.0, 1.0),
 ('O2', 'C1'): (58.0, 0.0, 1.0),
 ('O2', 'C2'): (40.0, 0.0, 1.0),
 ('O2', 'C3'): (44.0, 0.0, 1.0),
 ('O2', 'C4'): (63.0, 0.0, 1.0),
 ('O2', 'C5'): (52.0, 0.0, 1.0),
 ('O2', 'C6'): (47.0, 0.0, 1.0),
 ('O3', 'C1'): (36.0, 0.0, 1.0),
 ('O3', 'C2'): (62.0, 0.0, 1.0),
 ('O3', 'C3'): (39.0, 0.0, 1.0),
 ('O3', 'C4'): (51.0, 0.0, 1.0),
 ('O3', 'C5'): (58.0, 0.0, 1.0),
 ('O3', 'C6'): (66.0, 0.0, 1.0),
 ('O4', 'C1'): (49.0, 0.0, 1.0),
 ('O4', 'C2'): (43.0, 0.0, 1.0),
 ('O4', 'C3'): (57.0, 0.0, 1.0),
 ('O4', 'C4'): (45.0, 0.0, 1.0),
 ('O4', 'C5'): (41.0, 0.0, 1.0),
 ('O4', 'C6'): (53.0, 0.0, 1.0),
 ('O5', 'C1'): (64.0, 0.0, 1.0),
 ('O5', 'C2'): (51.0, 0.0, 1.0),
 ('O5', 'C3'): (46.0, 0.0, 1.0),
 ('O5', 'C4'): (59.0, 0.0, 1.0),
 ('O5', 'C5'): (37.0, 0.0, 1.0),
 ('O5', 'C6'): (42.0, 0.0, 1.0),
 ('O6', 'C

Nodos (12): ['O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6']
Arcos (36): red bipartita completa 6 x 6
Suma de los flujos exógenos (debe ser 0): 0.0


## Paso 3. El modelo base en Pyomo (variables continuas)

$$\min \; \sum_{(i,j)\in A} c_{ij}\,x_{ij}
\quad\text{s.a.}\quad
\sum_{j} x_{ij} - \sum_{k} x_{ki} = q_i \;\; \forall i \in N,
\qquad 0 \le x_{ij} \le 1$$

Las variables se declaran explícitamente continuas (`pyo.NonNegativeReals` con cota superior $1$) para verificar en el Paso 6 que la estructura de red garantiza una solución $0\text{–}1$ sin imponer integralidad.

In [7]:
import pyomo.environ as pyo

m = pyo.ConcreteModel(name="Instancia_D")
m.N = pyo.Set(initialize=list(N))
m.A = pyo.Set(initialize=list(A), dimen=2)

# Variables continuas en [0, 1] (sin imponer integralidad)
m.x = pyo.Var(
    m.A,
    domain=pyo.NonNegativeReals,
    bounds=lambda m, i, j: (A[(i, j)][1], A[(i, j)][2]),
)

m.obj = pyo.Objective(
    expr=sum(A[a][0] * m.x[a] for a in m.A),
    sense=pyo.minimize,
)

def balance(m, n):
    sale = sum(m.x[i, j] for (i, j) in m.A if i == n)
    entra = sum(m.x[i, j] for (i, j) in m.A if j == n)
    # Igualdad en todos los nodos: red balanceada (6 órdenes = 6 celdas, sum(q) = 0)
    return sale - entra == q[n]

m.bal = pyo.Constraint(m.N, rule=balance)

# Declaración del sufijo dual ANTES de resolver
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

print("Variables:", len(m.A), "| Restricciones nodales:", len(m.N))

Variables: 36 | Restricciones nodales: 12


## Paso 4. Resolución del modelo base (Pregunta 2)

Se resuelve el programa lineal continuo con HiGHS (`appsi_highs`), se muestra la asignación óptima, el tiempo total mínimo, el uso implícito del brazo compartido $C_1 + C_2$, y exportamos `resultados/solucion.csv` y `resultados/duales.csv` conforme a la estructura del repositorio.

In [8]:
res = pyo.SolverFactory("appsi_highs").solve(m)
print("Condición de término:", res.solver.termination_condition)
print(f"Valor óptimo base (LP continuo): {pyo.value(m.obj):.2f} minutos\n")

flujos = {a: pyo.value(m.x[a]) for a in m.A}
print(f"{'Orden':>8} -> {'Celda':<8} {'Flujo':>8} {'Tiempo (min)':>14}")
print("-" * 44)
filas_sol = []
uso_brazo_base = 0.0
for (o, c), v in sorted(flujos.items()):
    t_oc = A[(o, c)][0]
    filas_sol.append({"origen": o, "destino": c, "flujo": round(v, 6), "tiempo_min": t_oc})
    if v > 1e-6:
        print(f"{o:>8} -> {c:<8} {v:>8.4f} {t_oc:>14.2f}")
        if c in celdas_brazo:
            uso_brazo_base += t_oc * v

print("-" * 44)
print(f"Tiempo total óptimo:                 {pyo.value(m.obj):.2f} min")
print(f"Uso en celdas {celdas_brazo} (base): {uso_brazo_base:.2f} min (tope futuro = {tope_brazo:.2f} min)")

# Exportación reproducible a carpeta resultados/
out_dir = Path("resultados")
out_dir.mkdir(exist_ok=True)
pd.DataFrame(filas_sol).to_csv(out_dir / "solucion.csv", index=False, sep=";")
pd.DataFrame([
    {"nodo": n, "flujo_exogeno": q[n], "dual_min_por_orden": round(m.dual[m.bal[n]], 6)}
    for n in N
]).to_csv(out_dir / "duales.csv", index=False, sep=";")
print("\nArchivos exportados en resultados/solucion.csv y resultados/duales.csv")

Condición de término: optimal
Valor óptimo base (LP continuo): 236.00 minutos

   Orden -> Celda       Flujo   Tiempo (min)
--------------------------------------------
      O1 -> C4         1.0000          38.00
      O2 -> C3         1.0000          44.00
      O3 -> C1         1.0000          36.00
      O4 -> C2         1.0000          43.00
      O5 -> C5         1.0000          37.00
      O6 -> C6         1.0000          38.00
--------------------------------------------
Tiempo total óptimo:                 236.00 min
Uso en celdas ['C1', 'C2'] (base): 79.00 min (tope futuro = 77.00 min)

Archivos exportados en resultados/solucion.csv y resultados/duales.csv


## Paso 5. Verificación de la conservación de flujo y verificación independiente

Aquí se cumplen con las Exigencias comunes #2 y #3:
1. Comprobación numérica explícita de conservación de flujo ($\text{sale} - \text{entra} = q_n$) y respeto de cotas en cada nodo y arco.
2. Verificación independiente del óptimo por dos vías adicionales:
   * Vía independiente A: Algoritmo Húngaro (`scipy.optimize.linear_sum_assignment`).
   * Vía independiente B: Enumeración exhaustiva de las $6! = 720$ permutaciones posibles (que además certifica la unicidad del óptimo base y verificará luego el óptimo entero con la restricción adicional).

In [9]:
# 5.1 Verificación de conservación de flujo en todos los nodos
print(f"{'Nodo':<10}{'Sale':>10}{'Entra':>10}{'Neto':>10}{'Exigido':>10}   Estado")
print("-" * 58)
todo_ok = True
for n in N:
    sale = sum(v for (i, j), v in flujos.items() if i == n)
    entra = sum(v for (i, j), v in flujos.items() if j == n)
    neto = sale - entra
    ok = abs(neto - q[n]) < 1e-6
    todo_ok &= ok
    print(f"{n:<10}{sale:>10.2f}{entra:>10.2f}{neto:>10.2f}{q[n]:>10.2f}   {'OK' if ok else 'ERROR'}")

cotas_ok = all(A[a][1] - 1e-6 <= v <= A[a][2] + 1e-6 for a, v in flujos.items())
print("\nConservación de flujo verificada en todos los nodos:", todo_ok)
print("Cotas de capacidad [0, 1] respetadas en todos los arcos:", cotas_ok)

# 5.2 Verificación independiente del óptimo
from scipy.optimize import linear_sum_assignment
from itertools import permutations

matriz_costos = df_tiempos.set_index("orden")[celdas].to_numpy(dtype=float)

# Vía A: Algoritmo Húngaro
filas_h, cols_h = linear_sum_assignment(matriz_costos)
opt_hungaro = float(matriz_costos[filas_h, cols_h].sum())
asig_hungaro = [(ordenes[r], celdas[c], matriz_costos[r, c]) for r, c in zip(filas_h, cols_h)]

# Vía B: Enumeración exhaustiva de las 6! = 720 permutaciones
todas_base = []
todas_restringidas = []
for perm in permutations(range(len(celdas))):
    total = sum(matriz_costos[i, perm[i]] for i in range(len(ordenes)))
    uso_b = sum(matriz_costos[i, perm[i]] for i in range(len(ordenes)) if celdas[perm[i]] in celdas_brazo)
    asig = [(ordenes[i], celdas[perm[i]], matriz_costos[i, perm[i]]) for i in range(len(ordenes))]
    todas_base.append((total, uso_b, asig))
    if uso_b <= tope_brazo + 1e-9:
        todas_restringidas.append((total, uso_b, asig))

min_enum_base = min(x[0] for x in todas_base)
optimos_base = [x for x in todas_base if abs(x[0] - min_enum_base) < 1e-9]

print("\n=== VERIFICACIÓN INDEPENDIENTE (MODELO BASE) ===")
print(f"Pyomo + HiGHS (LP continuo):        {pyo.value(m.obj):.2f} min")
print(f"Algoritmo Húngaro (SciPy):          {opt_hungaro:.2f} min")
print(f"Enumeración exhaustiva (720 perm.): {min_enum_base:.2f} min (óptimos alternativos: {len(optimos_base)})")
assert abs(pyo.value(m.obj) - opt_hungaro) < 1e-6
assert abs(pyo.value(m.obj) - min_enum_base) < 1e-6
print("Resultado verificado por dos vías independientes: COINCIDENCIA EXACTA.")

Nodo            Sale     Entra      Neto   Exigido   Estado
----------------------------------------------------------
O1              1.00      0.00      1.00      1.00   OK
O2              1.00      0.00      1.00      1.00   OK
O3              1.00      0.00      1.00      1.00   OK
O4              1.00      0.00      1.00      1.00   OK
O5              1.00      0.00      1.00      1.00   OK
O6              1.00      0.00      1.00      1.00   OK
C1              0.00      1.00     -1.00     -1.00   OK
C2              0.00      1.00     -1.00     -1.00   OK
C3              0.00      1.00     -1.00     -1.00   OK
C4              0.00      1.00     -1.00     -1.00   OK
C5              0.00      1.00     -1.00     -1.00   OK
C6              0.00      1.00     -1.00     -1.00   OK

Conservación de flujo verificada en todos los nodos: True
Cotas de capacidad [0, 1] respetadas en todos los arcos: True

=== VERIFICACIÓN INDEPENDIENTE (MODELO BASE) ===
Pyomo + HiGHS (LP continuo):        23

## Paso 6. Integralidad y Total Unimodularidad (Pregunta 3)

### ¿Por qué resultaron enteros los flujos sin imponer integralidad?

Una matriz $M$ es totalmente unimodular (TU) si el determinante de toda submatriz cuadrada de $M$ pertenece al conjunto $\{0, 1, -1\}$.

* Propiedad de la matriz: La matriz de restricciones $M \in \{-1, 0, +1\}^{12 \times 36}$ del modelo base es la matriz de incidencia nodo–arco de un grafo dirigido (cada columna tiene exactamente un $+1$ en el nodo origen y un $-1$ en el nodo destino). Por teorema estándar de flujos en redes, toda matriz de incidencia nodo–arco de un grafo dirigido es totalmente unimodular.
* Condición sobre el lado derecho y las cotas: El vector de flujos exógenos $q \in \{+1, -1\}^{12}$ y las cotas de arco $(l_{ij}, u_{ij}) = (0, 1)$ son enteros.
* Consecuencia (Teorema de Hoffman–Kruskal): Todos los vértices (soluciones básicas factibles) del poliedro $\{x : Mx = q,\; 0 \le x \le 1\}$ tienen coordenadas enteras. Dado que el método símplex evalúa vértices, entrega automáticamente una solución óptima $0\text{–}1$ sin necesidad de ramificar.

A continuación se comprueba numéricamente que no existen flujos fraccionarios, construimos la matriz de incidencia nodo–arco $M$ y calculamos el determinante de cuatro submatrices cuadradas ($3 \times 3$ y $4 \times 4$), verificando que todos caen en $\{0, 1, -1\}$.

In [10]:
no_enteros = {a: v for a, v in flujos.items() if abs(v - round(v)) > 1e-6}
print("Cantidad de flujos no enteros en el modelo base:", len(no_enteros))
if no_enteros:
    print(no_enteros)

# Construcción explícita de la matriz de incidencia nodo-arco M (12 x 36)
arcos_lista = list(A.keys())
idx_nodo = {n: i for i, n in enumerate(N)}
idx_arco = {a: j for j, a in enumerate(arcos_lista)}

M = np.zeros((len(N), len(arcos_lista)), dtype=int)
for col, (o, c) in enumerate(arcos_lista):
    M[idx_nodo[o], col] = 1     # sale del nodo origen
    M[idx_nodo[c], col] = -1    # entra al nodo destino

print(f"\nDimensiones de la matriz de incidencia M: {M.shape} (rango = {np.linalg.matrix_rank(M)})")

submatrices_tu = [
    ("Submatriz 1 (3x3): filas [O1, O2, C1] x arcos [(O1,C1), (O2,C1), (O1,C2)]",
     ["O1", "O2", "C1"], [("O1", "C1"), ("O2", "C1"), ("O1", "C2")]),
    ("Submatriz 2 (3x3): filas [O1, O2, O3] x arcos óptimos [(O1,C4), (O2,C3), (O3,C1)]",
     ["O1", "O2", "O3"], [("O1", "C4"), ("O2", "C3"), ("O3", "C1")]),
    ("Submatriz 3 (4x4): ciclo bipartito [O1, O2, C1, C2] x [(O1,C1), (O1,C2), (O2,C1), (O2,C2)]",
     ["O1", "O2", "C1", "C2"], [("O1", "C1"), ("O1", "C2"), ("O2", "C1"), ("O2", "C2")]),
    ("Submatriz 4 (4x4): camino [O1, O2, C1, C3] x [(O1,C1), (O2,C1), (O2,C2), (O1,C3)]",
     ["O1", "O2", "C1", "C3"], [("O1", "C1"), ("O2", "C1"), ("O2", "C2"), ("O1", "C3")]),
]

print("\n=== VERIFICACIÓN DE DETERMINANTES EN SUBMATRICES DE M ===")
for titulo, filas_n, cols_a in submatrices_tu:
    S = M[np.ix_([idx_nodo[f] for f in filas_n], [idx_arco[a] for a in cols_a])]
    det = int(round(np.linalg.det(S)))
    assert det in (-1, 0, 1), f"Determinante fuera de {{-1, 0, 1}}: {det}"
    print(f"\n{titulo}")
    print(S)
    print(f"-> det(S) = {det}   [pertenece a {{-1, 0, 1}}]")

Cantidad de flujos no enteros en el modelo base: 0

Dimensiones de la matriz de incidencia M: (12, 36) (rango = 11)

=== VERIFICACIÓN DE DETERMINANTES EN SUBMATRICES DE M ===

Submatriz 1 (3x3): filas [O1, O2, C1] x arcos [(O1,C1), (O2,C1), (O1,C2)]
[[ 1  0  1]
 [ 0  1  0]
 [-1 -1  0]]
-> det(S) = 1   [pertenece a {-1, 0, 1}]

Submatriz 2 (3x3): filas [O1, O2, O3] x arcos óptimos [(O1,C4), (O2,C3), (O3,C1)]
[[1 0 0]
 [0 1 0]
 [0 0 1]]
-> det(S) = 1   [pertenece a {-1, 0, 1}]

Submatriz 3 (4x4): ciclo bipartito [O1, O2, C1, C2] x [(O1,C1), (O1,C2), (O2,C1), (O2,C2)]
[[ 1  1  0  0]
 [ 0  0  1  1]
 [-1  0 -1  0]
 [ 0 -1  0 -1]]
-> det(S) = 0   [pertenece a {-1, 0, 1}]

Submatriz 4 (4x4): camino [O1, O2, C1, C3] x [(O1,C1), (O2,C1), (O2,C2), (O1,C3)]
[[ 1  0  0  1]
 [ 0  1  1  0]
 [-1 -1  0  0]
 [ 0  0  0 -1]]
-> det(S) = -1   [pertenece a {-1, 0, 1}]


## Paso 7. Valores duales: unidades, convención de signos y rango de validez

### Convención de signos y unidades

En Pyomo con `appsi_highs` para un problema de minimización con restricciones $\text{sale} - \text{entra} = q_n$:

* Cada variable dual $\pi_n = \texttt{m.dual[m.bal[n]]}$ representa la tasa marginal $\frac{\partial z^*}{\partial q_n}$, medida en minutos por unidad de flujo exógeno (min/orden).
* Comprobación de dualidad fuerte: $\sum_{n \in N} \pi_n\, q_n = z^* = 236{,}00$ minutos, y para todo arco $(i, j) \in A$ se satisface factibilidad dual $\pi_i - \pi_j \le c_{ij}$ (con igualdad en los arcos óptimos activos, donde el costo reducido $\bar{c}_{ij} = c_{ij} - (\pi_i - \pi_j) = 0$).

### Peculiaridad estructural del problema de asignación y validez de los duales

1. Grado de libertad aditivo (red cerrada balanceada): Como $\sum_{n \in N} q_n = 0$, las 12 filas de $M$ son linealmente dependientes ($\text{rango}(M) = 11$). Sumar cualquier constante $k$ a todos los potenciales ($\pi_n' = \pi_n + k$) preserva exactamente $\sum \pi_n q_n = 236$ y las diferencias $\pi_i - \pi_j$. Por ello, el solver fija implícitamente un nodo de referencia (por ejemplo, $\pi_{C6} = 0$ min/orden) y los demás potenciales se interpretan relativos a ese nodo.
2. Rango de validez de perturbaciones individuales de $q_n$: En una asignación pura con igualdades estrictas, perturbar un único $q_n$ en $\pm \varepsilon$ rompe $\sum q_n = 0$ e inmediatamente vuelve el problema infactible (rango individual $[0, 0]$). Lo que sí es factible e interpretable económicamente es:
   * La diferencia de potenciales $\pi_i - \pi_j$: indica el tiempo imputado de asignar la orden $i$ a la celda $j$.
   * Los costos reducidos $\bar{c}_{ij} = c_{ij} - (\pi_i - \pi_j) \ge 0$ (en minutos/orden): indican cuánto debería reducirse el tiempo $c_{ij}$ de un arco no asignado para que resulte atractivo entrar a la solución óptima.
   * Degeneración primal: Toda solución básica tiene $6$ variables en $1$ y $5$ variables básicas en $0$, por lo que existen múltiples soluciones duales óptimas (incluso fijando $\pi_{C6} = 0$).
3. Dual de la restricción adicional (Paso 8): Cuando se agrega la restricción de brazo compartido $C_1 + C_2 \le T$, su dual $\lambda_{\text{brazo}} = \partial z^*_{LP}/\partial T = -1{,}00$ min de tiempo total / min de tope sí tiene interpretación directa y un rango de validez exacto $T \in [76, 79]$ minutos, verificado en el Paso 8.

In [11]:
print(f"{'Nodo':<8} {'Tipo':<10} {'q_n':>6} {'Dual pi_n (min/orden)':>24}")
print("-" * 54)
duales_base = {}
for n in N:
    d = float(m.dual[m.bal[n]])
    duales_base[n] = d
    tipo = "Orden" if n in ordenes else "Celda"
    print(f"{n:<8} {tipo:<10} {q[n]:>+6.0f} {d:>+24.4f}")

z_dual = sum(duales_base[n] * q[n] for n in N)
print("-" * 54)
print(f"Comprobación dualidad fuerte (sum pi_n * q_n): {z_dual:.4f} min (z* primal = {pyo.value(m.obj):.4f} min)")

# Matriz de costos reducidos c_bar_{ij} = c_{ij} - (pi_i - pi_j)
cr_df = pd.DataFrame(
    {c: [round(A[(o, c)][0] - (duales_base[o] - duales_base[c]), 4) for o in ordenes] for c in celdas},
    index=ordenes,
)
print("\nMatriz de costos reducidos c_bar[i, j] (min/asignación; 0 en los arcos óptimos):")
display(cr_df)

Nodo     Tipo          q_n    Dual pi_n (min/orden)
------------------------------------------------------
O1       Orden          +1                 +35.0000
O2       Orden          +1                 +34.0000
O3       Orden          +1                 +29.0000
O4       Orden          +1                 +37.0000
O5       Orden          +1                 +36.0000
O6       Orden          +1                 +38.0000
C1       Celda          -1                  -7.0000
C2       Celda          -1                  -6.0000
C3       Celda          -1                 -10.0000
C4       Celda          -1                  -3.0000
C5       Celda          -1                  -1.0000
C6       Celda          -1                  -0.0000
------------------------------------------------------
Comprobación dualidad fuerte (sum pi_n * q_n): 236.0000 min (z* primal = 236.0000 min)

Matriz de costos reducidos c_bar[i, j] (min/asignación; 0 en los arcos óptimos):


,C1,C2,C3,C4,C5,C6
O1,0.0,14.0,16.0,0.0,34.0,14.0
O2,17.0,0.0,0.0,26.0,17.0,13.0
O3,0.0,27.0,0.0,19.0,28.0,37.0
O4,5.0,0.0,10.0,5.0,3.0,16.0
O5,21.0,9.0,0.0,20.0,0.0,6.0
O6,0.0,16.0,2.0,3.0,16.0,0.0


## Paso 8. Restricción de brazo compartido (Preguntas 4, 5 y 6) y análisis de sensibilidad

### 8.1 Modelo extendido — Relajación lineal continua (Pregunta 4)

Se agrega la restricción leída dinámicamente desde `restriccion_adicional.csv`:

$$\sum_{i \in O} \sum_{j \in \{C_1, C_2\}} c_{ij}\,x_{ij} \le T \qquad (T = 77\text{ minutos})$$

manteniendo las variables continuas $x_{ij} \in [0, 1]$. Se verificará que:
1. El óptimo relajado sube de $236$ a $238{,}00$ minutos, saturando el tope ($77{,}00$ min).
2. Aparecen 6 variables fraccionarias con valores exactos $1/3$ y $2/3$ (correspondientes a combinar $1/3$ de la asignación óptima base y $2/3$ de la asignación entera restringida).
3. La matriz de restricciones ampliada deja de ser totalmente unimodular: Se muestra explícitamente la submatriz $6 \times 6$ asociada al ciclo fraccionario cuyo determinante es $-3 \notin \{-1, 0, 1\}$.

In [12]:
from fractions import Fraction

def construir_modelo_extendido(tope, binario=False):
    me = pyo.ConcreteModel()
    me.N = pyo.Set(initialize=list(N))
    me.A = pyo.Set(initialize=list(A), dimen=2)
    if binario:
        me.x = pyo.Var(me.A, domain=pyo.Binary)
    else:
        me.x = pyo.Var(
            me.A,
            domain=pyo.NonNegativeReals,
            bounds=lambda mod, i, j: (A[(i, j)][1], A[(i, j)][2]),
        )
    me.obj = pyo.Objective(expr=sum(A[a][0] * me.x[a] for a in me.A), sense=pyo.minimize)

    def bal_rule(mod, n):
        sale = sum(mod.x[i, j] for (i, j) in mod.A if i == n)
        entra = sum(mod.x[i, j] for (i, j) in mod.A if j == n)
        return sale - entra == q[n]

    me.bal = pyo.Constraint(me.N, rule=bal_rule)
    me.brazo = pyo.Constraint(
        expr=sum(A[(o, c)][0] * me.x[(o, c)] for o in ordenes for c in celdas_brazo) <= tope
    )
    if not binario:
        me.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return me

# --- Pregunta 4: LP continuo con restricción de brazo compartido ---
m_lp_ext = construir_modelo_extendido(tope=tope_brazo, binario=False)
res_lp_ext = pyo.SolverFactory("appsi_highs").solve(m_lp_ext)

z_lp_ext = pyo.value(m_lp_ext.obj)
flujos_lp_ext = {a: pyo.value(m_lp_ext.x[a]) for a in m_lp_ext.A}
uso_brazo_lp = sum(A[(o, c)][0] * flujos_lp_ext[(o, c)] for o in ordenes for c in celdas_brazo)
dual_brazo = float(m_lp_ext.dual[m_lp_ext.brazo])

print("=== PREGUNTA 4: MODELO EXTENDIDO (RELAJACIÓN LP CONTINUA) ===")
print("Condición de término:", res_lp_ext.solver.termination_condition)
print(f"Valor óptimo relajado (z_LP):      {z_lp_ext:.4f} minutos")
print(f"Uso del brazo {celdas_brazo}:      {uso_brazo_lp:.4f} / {tope_brazo:.2f} minutos (holgura = {tope_brazo - uso_brazo_lp:.4f} min)")
print(f"Dual de la restricción del brazo:  {dual_brazo:+.4f} min_total / min_tope\n")

print(f"{'Arco':<14} {'Flujo':>10} {'Fracción':>10} {'Tiempo (min)':>14}   Tipo")
print("-" * 62)
vars_fraccionarias = []
for a, v in sorted(flujos_lp_ext.items()):
    if v > 1e-6:
        es_frac = abs(v - round(v)) > 1e-6
        frac = str(Fraction(v).limit_denominator(1000))
        if es_frac:
            vars_fraccionarias.append((a, v, frac))
        print(f"{a[0]} -> {a[1]:<8} {v:>10.6f} {frac:>10} {A[a][0]:>14.2f}   {'FRACCIONARIA' if es_frac else 'Entera'}")

print(f"\nTotal de variables fraccionarias: {len(vars_fraccionarias)}")

# Demostración numérica de la pérdida de Total Unimodularidad en la matriz ampliada
fila_brazo = np.array([[A[a][0] if a[1] in celdas_brazo else 0.0 for a in arcos_lista]])
M_ext = np.vstack([M.astype(float), fila_brazo])

filas_ciclo = [idx_nodo[n] for n in ["O2", "O4", "O5", "C2", "C3"]] + [len(N)]
cols_ciclo = [idx_arco[a] for a in [("O2", "C2"), ("O2", "C3"), ("O4", "C2"), ("O4", "C5"), ("O5", "C3"), ("O5", "C5")]]
S_no_tu = M_ext[np.ix_(filas_ciclo, cols_ciclo)]
det_no_tu = np.linalg.det(S_no_tu)

print("\n=== DEMOSTRACIÓN DE PÉRDIDA DE TU (SUBMATRIZ 6x6 DEL CICLO FRACCIONARIO) ===")
print("Filas: [O2, O4, O5, C2, C3, Restricción_C1+C2]")
print("Columnas: [(O2,C2), (O2,C3), (O4,C2), (O4,C5), (O5,C3), (O5,C5)]")
print(S_no_tu)
print(f"-> det(S_no_tu) = {det_no_tu:.4f} (|det| = {abs(round(det_no_tu))} != 0, 1 -> NO ES TU)")

=== PREGUNTA 4: MODELO EXTENDIDO (RELAJACIÓN LP CONTINUA) ===
Condición de término: optimal
Valor óptimo relajado (z_LP):      238.0000 minutos
Uso del brazo ['C1', 'C2']:      77.0000 / 77.00 minutos (holgura = 0.0000 min)
Dual de la restricción del brazo:  -1.0000 min_total / min_tope

Arco                Flujo   Fracción   Tiempo (min)   Tipo
--------------------------------------------------------------
O1 -> C4         1.000000          1          38.00   Entera
O2 -> C2         0.666667        2/3          40.00   FRACCIONARIA
O2 -> C3         0.333333        1/3          44.00   FRACCIONARIA
O3 -> C1         1.000000          1          36.00   Entera
O4 -> C2         0.333333        1/3          43.00   FRACCIONARIA
O4 -> C5         0.666667        2/3          41.00   FRACCIONARIA
O5 -> C3         0.666667        2/3          46.00   FRACCIONARIA
O5 -> C5         0.333333        1/3          37.00   FRACCIONARIA
O6 -> C6         1.000000          1          38.00   Entera

Tot

### 8.2 Modelo extendido — Variables binarias y brecha de integralidad (Pregunta 5)

Se resuelve el mismo modelo declarando las variables como binarias ($x_{ij} \in \{0, 1\}$), calculamos la brecha de integralidad respecto de la relajación lineal y realizamos la verificación independiente contra las $720$ permutaciones factibles.

In [13]:
# --- Pregunta 5: Analisis con variables binarias ---
m_milp = construir_modelo_extendido(tope=tope_brazo, binario=True)
res_milp = pyo.SolverFactory("appsi_highs").solve(m_milp)

z_ip = pyo.value(m_milp.obj)
flujos_ip = {a: pyo.value(m_milp.x[a]) for a in m_milp.A}
uso_brazo_ip = sum(A[(o, c)][0] * flujos_ip[(o, c)] for o in ordenes for c in celdas_brazo)

brecha_abs = z_ip - z_lp_ext
brecha_rel_lp = (brecha_abs / z_lp_ext) * 100.0
brecha_rel_ip = (brecha_abs / z_ip) * 100.0

print("=== PREGUNTA 5: MODELO EXTENDIDO ===")
print("Condición de término:", res_milp.solver.termination_condition)
print(f"Valor óptimo entero (z_IP):        {z_ip:.2f} minutos")
print(f"Uso del brazo {celdas_brazo}:      {uso_brazo_ip:.2f} / {tope_brazo:.2f} minutos (holgura = {tope_brazo - uso_brazo_ip:.2f} min)\n")

print(f"{'Orden':>8} -> {'Celda':<8} {'x_ij':>6} {'Tiempo (min)':>14}")
print("-" * 42)
for (o, c), v in sorted(flujos_ip.items()):
    if v > 0.5:
        print(f"{o:>8} -> {c:<8} {int(round(v)):>6d} {A[(o, c)][0]:>14.2f}")

print("\n=== BRECHA DE INTEGRALIDAD ===")
print(f"Relajación lineal (z_LP):          {z_lp_ext:.4f} min")
print(f"Óptimo entero (z_IP):              {z_ip:.4f} min")
print(f"Brecha absoluta (z_IP - z_LP):     {brecha_abs:.4f} min")
print(f"Brecha relativa vs LP:             {brecha_rel_lp:.4f}%")
print(f"Brecha relativa vs IP (MIP gap):   {brecha_rel_ip:.4f}%")

# Verificación independiente por enumeración exhaustiva (del Paso 5)
min_enum_restr = min(x[0] for x in todas_restringidas)
optimos_restr = [x for x in todas_restringidas if abs(x[0] - min_enum_restr) < 1e-9]
print(f"\nVerificación independiente (enumeración exhaustiva): {min_enum_restr:.2f} min (óptimos alternativos: {len(optimos_restr)})")
assert abs(z_ip - min_enum_restr) < 1e-6

=== PREGUNTA 5: MODELO EXTENDIDO ===
Condición de término: optimal
Valor óptimo entero (z_IP):        239.00 minutos
Uso del brazo ['C1', 'C2']:      76.00 / 77.00 minutos (holgura = 1.00 min)

   Orden -> Celda      x_ij   Tiempo (min)
------------------------------------------
      O1 -> C4            1          38.00
      O2 -> C2            1          40.00
      O3 -> C1            1          36.00
      O4 -> C5            1          41.00
      O5 -> C3            1          46.00
      O6 -> C6            1          38.00

=== BRECHA DE INTEGRALIDAD ===
Relajación lineal (z_LP):          238.0000 min
Óptimo entero (z_IP):              239.0000 min
Brecha absoluta (z_IP - z_LP):     1.0000 min
Brecha relativa vs LP:             0.4202%
Brecha relativa vs IP (MIP gap):   0.4184%

Verificación independiente (enumeración exhaustiva): 239.00 min (óptimos alternativos: 1)


### 8.3 Análisis de sensibilidad (`resolver()` y barrido paramétrico del tope)

Se utiliza la función `resolver()` de la plantilla para dos análisis complementarios:
1. Sensibilidad del tope del brazo compartido $T \in [74, 82]$ minutos (comparando LP vs. MILP) para determinar empíricamente el rango de validez exacto del valor dual $\lambda_{\text{brazo}} = -1{,}00$ min/min ($T \in [76, 79]$) y mostrar la discontinuidad escalonada del óptimo entero.
2. Sensibilidad operativa del modelo base prohibiendo cada uno de los 6 arcos óptimos ($u_{ij} = 0$) para cuantificar el margen de robustez de cada asignación y el sobrecosto mínimo ante indisponibilidad de una celda para una orden.

In [14]:
def resolver(A_mod, q_mod=None):
    """Resuelve una variante del modelo base y devuelve (valor óptimo, flujos)."""
    q2 = q_mod if q_mod is not None else q
    mm = pyo.ConcreteModel()
    mm.A = pyo.Set(initialize=list(A_mod), dimen=2)
    mm.N = pyo.Set(initialize=list(q2))
    mm.x = pyo.Var(
        mm.A,
        domain=pyo.NonNegativeReals,
        bounds=lambda mod, i, j: (A_mod[(i, j)][1], A_mod[(i, j)][2]),
    )
    mm.obj = pyo.Objective(expr=sum(A_mod[a][0] * mm.x[a] for a in mm.A), sense=pyo.minimize)

    def bal(mod, n):
        sale = sum(mod.x[i, j] for (i, j) in mod.A if i == n)
        entra = sum(mod.x[i, j] for (i, j) in mod.A if j == n)
        return sale - entra == q2[n]

    mm.bal = pyo.Constraint(mm.N, rule=bal)
    # Evitamos fallar si es infactible usando load_solutions=False si fuese necesario,
    # o capturando el estado de terminación de manera segura.
    opt = pyo.SolverFactory("appsi_highs")
    r = opt.solve(mm, load_solutions=False)
    if str(r.solver.termination_condition) != "optimal":
        return None, None
    mm.solutions.load_from(r)
    return pyo.value(mm.obj), {a: pyo.value(mm.x[a]) for a in mm.A}

# 1) Sensibilidad paramétrica del tope T del brazo compartido (LP vs MILP)
filas_sens = []
solver_opt = pyo.SolverFactory("appsi_highs")

for T in range(74, 83):
    m_l = construir_modelo_extendido(tope=float(T), binario=False)
    # Usamos load_solutions=False para evitar excepciones si es infactible
    r_l = solver_opt.solve(m_l, load_solutions=False)
    ok_l = str(r_l.solver.termination_condition) == "optimal"
    if ok_l:
        m_l.solutions.load_from(r_l)

    m_i = construir_modelo_extendido(tope=float(T), binario=True)
    r_i = solver_opt.solve(m_i, load_solutions=False)
    ok_i = str(r_i.solver.termination_condition) == "optimal"
    if ok_i:
        m_i.solutions.load_from(r_i)

    z_l = round(pyo.value(m_l.obj), 4) if ok_l else None
    d_l = round(float(m_l.dual[m_l.brazo]), 4) if ok_l else None
    z_i = round(pyo.value(m_i.obj), 4) if ok_i else None
    uso_i = round(sum(A[(o, c)][0] * pyo.value(m_i.x[(o, c)]) for o in ordenes for c in celdas_brazo), 2) if ok_i else None
    gap = round(z_i - z_l, 4) if (ok_l and ok_i) else None

    filas_sens.append({
        "Tope_T_min": T,
        "z_LP_min": z_l if ok_l else "Infactible",
        "Dual_LP_min_por_min": d_l if ok_l else "-",
        "z_MILP_min": z_i if ok_i else "Infactible",
        "Uso_brazo_MILP": uso_i if ok_i else "-",
        "Brecha_min": gap if gap is not None else "-",
    })

print("=== SENSIBILIDAD DEL TOPE DEL BRAZO C1+C2 (RANGO DE VALIDEZ DEL DUAL) ===")
display(pd.DataFrame(filas_sens))

# 2) Sensibilidad en el modelo base: prohibir cada arco óptimo con resolver()
arcos_optimos_base = [a for a, v in sorted(flujos.items()) if v > 0.5]
z_base = pyo.value(m.obj)
print("\n=== SENSIBILIDAD BASE: PROHIBIR CADA ARCO ÓPTIMO (u_ij = 0) ===")
filas_arco = []
for a_ban in arcos_optimos_base:
    A_mod = dict(A)
    A_mod[a_ban] = (A[a_ban][0], 0.0, 0.0)
    z_alt, _ = resolver(A_mod)
    inc = z_alt - z_base if z_alt is not None else None
    filas_arco.append({
        "Arco_prohibido": f"{a_ban[0]} -> {a_ban[1]}",
        "Tiempo_actual_min": A[a_ban][0],
        "Nuevo_optimo_min": z_alt if z_alt is not None else "Infactible",
        "Sobrecosto_min": inc if inc is not None else "-",
        "Margen_aumento_tiempo_min": f"+{inc:.0f} (hasta {A[a_ban][0]+inc:.0f} min)" if inc is not None else "-",
    })
display(pd.DataFrame(filas_arco))

=== SENSIBILIDAD DEL TOPE DEL BRAZO C1+C2 (RANGO DE VALIDEZ DEL DUAL) ===


,Tope_T_min,z_LP_min,Dual_LP_min_por_min,z_MILP_min,Uso_brazo_MILP,Brecha_min
0,74,Infactible,-,Infactible,-,-
1,75,Infactible,-,Infactible,-,-
2,76,239.0,-1.0,239.0,76.0,0.0
3,77,238.0,-1.0,239.0,76.0,1.0
4,78,237.0,-1.0,239.0,76.0,2.0
5,79,236.0,-0.0,236.0,79.0,0.0
6,80,236.0,-0.0,236.0,79.0,0.0
7,81,236.0,-0.0,236.0,79.0,0.0
8,82,236.0,-0.0,236.0,79.0,0.0



=== SENSIBILIDAD BASE: PROHIBIR CADA ARCO ÓPTIMO (u_ij = 0) ===


,Arco_prohibido,Tiempo_actual_min,Nuevo_optimo_min,Sobrecosto_min,Margen_aumento_tiempo_min
0,O1 -> C4,38.0,241.0,5.0,+5 (hasta 43 min)
1,O2 -> C3,44.0,239.0,3.0,+3 (hasta 47 min)
2,O3 -> C1,36.0,241.0,5.0,+5 (hasta 41 min)
3,O4 -> C2,43.0,239.0,3.0,+3 (hasta 46 min)
4,O5 -> C5,37.0,239.0,3.0,+3 (hasta 40 min)
5,O6 -> C6,38.0,245.0,9.0,+9 (hasta 47 min)


## Paso 9. Respuestas integradas a las preguntas de la Instancia D y limitaciones del modelo

### Síntesis de respuestas (Equipo D — Industria 4.0)

1. Formulación como flujo a costo mínimo (Pregunta 1): El problema se modela sobre una red bipartita completa $G = (O \cup C, O \times C)$ con 12 nodos (6 órdenes con flujo exógeno $q_i = +1$ orden y 6 celdas con $q_j = -1$ orden) y 36 arcos con costo $c_{ij}$ en minutos y capacidad $[0, 1]$. Todas las restricciones son igualdades porque $\sum q_n = 0$ y la asignación debe ser biyectiva (una orden por celda).
2. Resolución continua del modelo base (Pregunta 2): El tiempo mínimo total es $236$ minutos (solución única, verificada mediante el Algoritmo Húngaro y enumeración exhaustiva de las 720 permutaciones), con asignación:
   * $O_1 \to C_4$ ($38$ min), $O_2 \to C_3$ ($44$ min), $O_3 \to C_1$ ($36$ min), $O_4 \to C_2$ ($43$ min), $O_5 \to C_5$ ($37$ min), $O_6 \to C_6$ ($38$ min).
   * En esta solución, las celdas $C_1$ y $C_2$ suman $36 + 43 = 79$ minutos, excediendo en $2$ minutos el tope posterior de $77$ minutos.
3. Integralidad automática y Total Unimodularidad (Pregunta 3): Aunque las variables se declararon continuas en $[0, 1]$, todos los flujos resultaron $0\text{–}1$ porque la matriz de incidencia nodo–arco $M$ es totalmente unimodular (toda submatriz cuadrada tiene determinante en $\{-1, 0, 1\}$, comprobado numéricamente en cuatro submatrices de $M$) y el lado derecho $q$ junto con las cotas $[0, 1]$ son enteros. En consecuencia, todos los vértices del poliedro son enteros.
4. Relajación lineal con restricción de brazo compartido (Pregunta 4): Al incorporar $\sum_{i \in O} (c_{i,C1}x_{i,C1} + c_{i,C2}x_{i,C2}) \le 77$, el valor óptimo relajado sube a $238{,}00$ minutos, saturando el tope en exactamente $77{,}00$ minutos (con dual $\lambda_{\text{brazo}} = -1{,}00$ min/min válido en $T \in [76, 79]$ min). Emergen 6 variables fraccionarias:
   * $x_{O2,C2} = 2/3$, $x_{O2,C3} = 1/3$, $x_{O4,C2} = 1/3$, $x_{O4,C5} = 2/3$, $x_{O5,C3} = 2/3$, $x_{O5,C5} = 1/3$ (manteniendo $x_{O1,C4} = x_{O3,C1} = x_{O6,C6} = 1$).
   * Geométricamente, esta solución fraccionaria es exactam nte la combinación convexa $\frac{1}{3} x^*_{\text{base}} + \frac{2}{3} x^*_{\text{entero}}$ ($\frac{1}{3}\cdot 236 + \frac{2}{3}\cdot 239 = 238$ min; $\frac{1}{3}\cdot 79 + \frac{2}{3}\cdot 76 = 77$ min), impracticable físicamente en planta porque no se pueden fraccionar órdenes entre dos celdas.
5. Óptimo binario y brecha de integralidad (Pregunta 5): Al imponer $x_{ij} \in \{0, 1\}$, el óptimo entero único es $239$ minutos (verificado contra las 720 permutaciones), con asignación:
   * $O_1 \to C_4$ ($38$ min), $O_2 \to C_2$ ($40$ min), $O_3 \to C_1$ ($36$ min), $O_4 \to C_5$ ($41$ min), $O_5 \to C_3$ ($46$ min), $O_6 \to C_6$ ($38$ min).
   * El uso real del brazo compartido es $36 + 40 = 76$ minutos (deja $1$ minuto de holgura respecto al tope de $77$ min).
   * La brecha de integralidad es de $1$ minuto en términos absolutos, equivalente a $0{,}4202\%$ respecto a la relajación lineal ($z^*_{LP} = 238$) y $0{,}4184\%$ respecto al óptimo entero ($z^*_{IP} = 239$). Además, el análisis de sensibilidad demuestra que, aunque el dual continuo promete ahorrar $1$ min por cada minuto extra de tope en $[76, 79]$, en el mundo discreto subir el tope de $77$ a $78$ minutos produce $0$ ahorro real ($z^*_{IP}$ permanece en $239$ min hasta alcanzar $T = 79$ min, donde salta a $236$ min).
6. Propiedad perdida, ramificación y conexión con la Unidad 2 (Pregunta 6): Al agregar una única restricción lateral ponderada por los tiempos de proceso $c_{ij}$ (que ya no tiene coeficientes en $\{-1, 0, +1\}$ ni conserva la estructura de incidencia nodo–arco), se pierde la Total Unimodularidad de la matriz de restricciones —como se demostró exhibiendo una submatriz $6 \times 6$ con $|\det| = 3$—. Por regla de Cramer ($x_B = B^{-1}b$), dividir por determinantes distintos de $\pm 1$ introduce vértices fraccionarios en el poliedro relajado, de modo que el óptimo del LP ya no coincide con el óptimo entero.

---

### Limitación principal del modelo (Exigencia común #5)

* Supuesto incumplido en la realidad industrial: El modelo asume que los tiempos de preparación y proceso son puramente aditivos, independientes y libres de interferencia temporal, y representa el recurso compartido entre $C_1$ y $C_2$ (un único brazo de carga) únicamente como una cota agregada de suma de tiempos ($t_{C1} + t_{C2} \le 77$ min), omitiendo la secuenciación en el tiempo, el bloqueo/espera por concurrencia física (cuando ambas celdas requieren el brazo simultáneamente para carga o descarga) y la variabilidad estocástica propia de sensores, microparadas y calibraciones en celdas robotizadas.
* Dirección del sesgo sobre la conclusión: Esta omisión subestima sistemáticamente el tiempo real de ejecución y sobreestima la capacidad efectiva del brazo compartido. En operación real, los tiempos muertos por espera del brazo en $C_1$ y $C_2$ harán que el tiempo total consumido sea estrictamente mayor que los $239$ minutos reportados, e incluso una asignación que en el modelo cumple holgadamente $76 \le 77$ minutos podría experimentar congestión o sobrepasar la ventana del turno si las solicitudes de carga coinciden en el tiempo.